# Web Scraping - Indeed.com
General steps for Web Scraping
1. Check whether the website allows web scraping
2. Obtain the source code (HTML File) by using the website URL
3. Download the website content
4. Parse the content using keywords tags for elements of interest
5. Extract relevant data/features
6. Organize raw data in structured format (e.g., CSV)

### Import Dependencies 

In [1]:
!pip install arsenic
!pip install structlog

In [2]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from datetime import datetime
from arsenic import get_session
from arsenic.browsers import Firefox
from arsenic.services import Geckodriver
import asyncio

# Disable arsenic logging to stdout:
import structlog
import logging

logger = logging.getLogger()
logger.setLevel(logging.WARN)
structlog.configure(logger_factory=lambda: logger)

# For iterating several positions and locations:
import itertools

### Path to webdriver (Firefox, Chrome) 

In [3]:
# Ensure that the driver path is correct before running this script.
# Microsoft Windows:
driver_path = 'D:/geckodriver/geckodriver.exe'

options = {
    'moz:firefoxOptions': {
        # if you want it to be headless
        'args': ['-headless'],
        'log': {
            'level': 'warn'
        },
        # Needed for windows / non-default firefox install
        'binary': 'C:/Program Files/Mozilla Firefox/firefox.exe'
    }
}

### Define position and location 

In [4]:
positions = ["data scientist", "manager of analytics"]
locations = ["CA"]


def get_url(position, location):
    url_template = "https://www.indeed.com/jobs?q={}&l={}"
    url = url_template.format(position, location)
    return url


urls = [get_url(p, l) for p, l in itertools.product(positions, locations)]

# Define dataframe for storing scraping results:
dataframe = pd.DataFrame(columns=["Title", "Company", "Location", "Rating", "Date", "Salary", "Description", "Links"])

### Scrape job postings

In [ ]:
## Number of postings to scrape:
postings = 500

## Number of browser instances to use:
n = 3

pages = list(range(0, postings, 10))

state = {
  'lock': asyncio.Lock(),
  'ids': set(),
  'n': 0
}

for url in urls:
    df = pd.DataFrame(columns=["Title", "Company", "Location", "Rating", "Date", "Salary", "Description", "Links"])
    async def get_jobs(url, pages, state):
      data = []
      async with get_session(Geckodriver(binary=driver_path, log_file=asyncio.subprocess.PIPE), Firefox(**options)) as session:
        for i in pages:
          await session.get(url + "&start=" + str(i))
          jobs = await session.get_elements("[class='job_seen_beacon']")

          for job in jobs:
            result_html = await job.get_property('innerHTML')
            soup = BeautifulSoup(result_html, 'html.parser')

            liens = await job.get_elements("a")
            link = await liens[0].get_attribute("href")

            title = soup.select('.jobTitle')[0].get_text().strip()
            try:
              company = soup.select('.companyName')[0].get_text().strip()
            except:
              continue
            location = soup.select('.companyLocation')[0].get_text().strip()
            try:
                salary = soup.select('.salary-snippet-container')[0].get_text().strip()
            except:
                salary = 'NaN'
            try:
                rating = soup.select('.ratingNumber')[0].get_text().strip()
            except:
                rating = 'NaN'
            try:
                date = soup.select('.date')[0].get_text().strip()
            except:
                date = 'NaN'
            try:
                description = soup.select('.job-snippet')[0].get_text().strip()
            except:
                description = ''

            Id = f"{title}{company}{location}{rating}{date}{salary}{description}"
            dupe = False
            async with state['lock']:
              if Id in state['ids']:
                dupe = True
              else:
                state['ids'].add(Id)
                state['n'] = state['n'] + 1
                print("Job number {0:4d} added - {1:s}".format(state['n'],title))
            if dupe:
              continue

            data.append({
              'Title': title,
              "Company": company,
              'Location': location,
              'Rating': rating,
              'Date': date,
              "Salary": salary,
              "Description": description,
              "Links": link
            })

            # print("Job number {0:4d} added - {1:s}".format(jn,title))
          i = i + 10
      return data

    tasks = [asyncio.create_task(get_jobs(url, p, state)) for p in np.array_split(pages, n)]
    df = pd.DataFrame([j for task in tasks for j in await task])
    dataframe = pd.concat([dataframe, df], ignore_index=True)

Job number    1 added - Bravo-Sr Data Analyst - Remote
Job number    2 added - SOLUTIONS ARCHITECT - DATA & ANALYTICS
Job number    3 added - Junior Data Scientist
Job number    4 added - Junior Data Scientist
Job number    5 added - Bravo-Sr Data Analyst - Remote
Job number    6 added - SOLUTIONS ARCHITECT - DATA & ANALYTICS
Job number    7 added - Advanced Distribution Management System Lead consultant (ADMS)
Job number    8 added - Team Leader - Walmart/Sams Club
Job number    9 added - IT Project Manager
Job number   10 added - Senior Manager IT-Finance
Job number   11 added - Advanced Distribution Management System Lead consultant (ADMS)
Job number   12 added - IT Project Manager


### Scrape full job descriptions

In [6]:
Links_list = dataframe['Links'].tolist()

import random
import time

async def get_description(urls):
  descriptions = []
  async with get_session(Geckodriver(binary=driver_path, log_file=asyncio.subprocess.PIPE), Firefox(**options)) as session:
    for url in urls:
      await session.get("https://www.indeed.com"+url)
      # Add a random delay:
      time.sleep(random.uniform(0.5, 1.5))
      jd = await session.get_element('#jobDescriptionText')
      descriptions.append(await jd.get_text())
      await asyncio.sleep(random.random() * 1.5)
  return descriptions

## Number of browser instances to use:
n = 3

tasks = [asyncio.create_task(get_description(urls)) for urls in np.array_split(Links_list, n)]
dataframe['Descriptions'] = [desc for task in tasks for desc in await task]

### Save results

In [8]:
# Convert the dataframe to a csv file:
dataframe.to_csv("webscraping_results_assignmnet3.csv", index=False)

In [9]:
dataframe

,Title,Company,Location,Rating,Date,Salary,Description,Links,Descriptions
0,Data Scientist,Great American Insurance Company,"Remote in Los Angeles, CA 90017",3.8,PostedPosted 6 days ago,"$95,000 - $112,000 a year",Experience: 2+ years of experience with struct...,/rc/clk?jk=7e518ef57fbbcc33&fccid=11e7471668f8...,Be Here. Be Great. Working for a leader in the...
1,Data Scientist,ironSource,"Remote in San Francisco, CA 94107",NaN,PostedPosted 30+ days ago,"$140,000 - $155,000 a year",Develop and deploy machine learning models for...,/rc/clk?jk=499a147eb89f9a8b&fccid=1db73ef8e1dc...,Gather and analyze large sets of structured an...
2,Sr. Data Scientist,Adobe,"San Francisco, CA 94103 (South Of Market area)",4.3,PostedPosted 14 days ago,"$122,400 - $223,000 a year",Drive analytics for user revenue/engagement gr...,/rc/clk?jk=7e4bb4278d9c80cc&fccid=f89deb5a97c7...,Our Company\n\nChanging the world through digi...
3,Data Scientist,Spindl,"Remote in San Francisco, CA",NaN,PostedPosted 7 days ago,NaN,You’ll be the founding data scientist in a wel...,/rc/clk?jk=0741426907225469&fccid=dd616958bd9d...,You’ll be the founding data scientist in a wel...
4,Educational Data Scientist,Expatiate Communications,"Pasadena, CA 91101 (Pasadena area)",3.6,EmployerActive 4 days ago,"$100,000 - $150,000 a year",Knowledge and deployment of advanced statistic...,/pagead/clk?mo=r&ad=-6NYlbfkN0AnQhcI30S35noZYX...,The AI in Education at Expatiate Communication...
...,...,...,...,...,...,...,...,...,...
1056,Senior Product Manager I,BOLD,"Remote in San Francisco, CA",3.5,PostedPosted 30+ days ago,"$145,000 - $189,000 a year",You will analyze data and analytics to determi...,/rc/clk?jk=916bba8dfb485f12&fccid=b4ae81341320...,ABOUT THIS JOB\n\nWe are looking for a Product...
1057,Big Data Manager- Hybrid,Mantek Solutions,"Hybrid remote in Costa Mesa, CA",NaN,PostedPosted 30+ days ago,"$175,000 - $190,000 a year",Identify proper analytic methodology and ensur...,/rc/clk?jk=4c3342dad7164b24&fccid=d14958d1be85...,Big Data Manager- Hybrid - https://ace.wd5.myw...
1058,IT Healthcare Project Manager (IT Business Sys...,County of Riverside,"Moreno Valley, CA 92555 (La Jolla area)",3.8,PostedPosted 30+ days ago,NaN,The department is looking for an individual th...,/rc/clk?jk=84489dbe065bdcc9&fccid=bd8753bffa1f...,The County of Riverside's University Health Sy...
1059,Senior Product Manager - Digital Customer Expe...,Siemens,"Remote in Fremont, CA 94538",4.0,PostedPosted 30+ days ago,"$136,700 - $246,100 a year",7+ years as a product manager and/or equivalen...,/rc/clk?jk=fc5087cd7b39bfd4&fccid=ea6bb53f0b18...,Position Summary\nAre you eager to be at the f...
